# Практична робота №1: Застосування алгоритмів класифікації та кластеризації (РІШЕННЯ)

**Мета роботи:** Ознайомитися з базовими етапами підготовки даних, завантаженням стандартних датасетів та навчанням найпростіших моделей класифікації (Logistic Regression, Decision Tree, Multilayer Perceptron) та кластеризації (K-Means) за допомогою бібліотек `scikit-learn`, `pandas`, `numpy` та `tensorflow`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Scikit-learn: датасети, розділення, моделі класифікації та кластеризації
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score

# TensorFlow / Keras: для створення простого MLP
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

print("Усі бібліотеки успішно імпортовано!")

## Частина 1. Класифікація (Supervised Learning)

Працюємо з класичним датасетом **Iris** (квіти ірису).

In [ ]:
# 1. Завантаження датасету
iris = load_iris()
X = iris.data
y = iris.target

# Перетворимо в DataFrame для зручності перегляду
df = pd.DataFrame(X, columns=iris.feature_names)
df['target'] = y

print("Перші 5 рядків датасету:")
print(df.head())

# Виведення інформації про класи
print("\nКласи в датасеті:")
for label, name in enumerate(iris.target_names):
    print(f"  Мітка {label}: {name}")

# Розділення на навчальну та тестову вибірки (80% / 20%)
# test_size=0.2 — залишаємо 20% даних для перевірки якості
# random_state=42 — фіксуємо зерно для відтворюваності розбиття
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Нормалізація ознак (Z-score normalization)
scaler = StandardScaler()
# fit_transform обчислює mean та std на X_train і масштабує його
X_train_scaled = scaler.fit_transform(X_train)
# transform використовує mean та std з X_train для X_test (уникаємо data leakage)
X_test_scaled = scaler.transform(X_test)

print(f"\nРозмір навчальної вибірки: {X_train.shape}")
print(f"Розмір тестової вибірки: {X_test.shape}")

In [ ]:
# --- ЗАВДАННЯ 1: Logistic Regression ---
# LogisticRegression за замовчуванням використовує L2-регуляризацію та solver='lbfgs'
model_lr = LogisticRegression()

# Навчаємо на scaled даних, бо лінійні моделі чутливі до масштабу ознак
model_lr.fit(X_train_scaled, y_train)

# Отримуємо передбачення для тестової вибірки
y_pred_lr = model_lr.predict(X_test_scaled)

print("Accuracy (Logistic Regression):", accuracy_score(y_test, y_pred_lr))

In [ ]:
# --- ЗАВДАННЯ 2: Decision Tree ---
# random_state=42 фіксує вибір розбиттів у вузлах
model_dt = DecisionTreeClassifier(random_state=42)

# Навчаємо на НЕНОРМАЛІЗОВАНИХ даних (X_train),
# оскільки дерева рішень роблять розбиття за пороговими значеннями (x_i > threshold)
# і масштабування на них не впливає
model_dt.fit(X_train, y_train)

y_pred_dt = model_dt.predict(X_test)

print("Accuracy (Decision Tree):", accuracy_score(y_test, y_pred_dt))

In [ ]:
# --- ЗАВДАННЯ 3: Multilayer Perceptron (Keras) ---
model_mlp = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(8, activation='relu'),
    # Вихідний шар: 3 нейрони (3 класи), activation='softmax' дає розподіл ймовірностей
    Dense(3, activation='softmax')
])

# sparse_categorical_crossentropy використовується, коли target надано цілими числами (0, 1, 2)
model_mlp.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model_mlp.fit(X_train_scaled, y_train, epochs=30, batch_size=8, verbose=0)

loss, acc = model_mlp.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Accuracy (MLP Keras): {acc:.4f}")

## Частина 2. Кластеризація (Unsupervised Learning)

In [ ]:
# --- ЗАВДАННЯ 4: K-Means ---
# n_clusters=3 — вказуємо шукати 3 класи/групи
# random_state=42 — фіксуємо вибір початкових центроїдів
kmeans = KMeans(n_clusters=3, random_state=42)

# fit_predict обчислює центроїди та повертає мітку кластера для кожного об'єкта
cluster_labels = kmeans.fit_predict(X_train_scaled)

plt.figure(figsize=(8, 5))
plt.scatter(X_train_scaled[:, 0], X_train_scaled[:, 1], c=cluster_labels, cmap='viridis', edgecolor='k')
plt.title("Результат кластеризації K-Means (перші 2 ознаки)")
plt.xlabel("Sepal length (scaled)")
plt.ylabel("Sepal width (scaled)")
plt.grid(True)
plt.show()

## Контрольні запитання:
1. **Чому для Logistic Regression та MLP важливе масштабування, а для Decision Tree — ні?**
   *Відповідь:* Лінійні моделі та нейромережі використовують сумування з вагами та градієнтний спуск, тому ознаки з більшим масштабом переважатимуть. Дерева рішень роблять розбиття за пороговими значеннями для кожної ознаки окремо, тому масштаб значення не має.

2. **Яка функція активації використовується на вихідному шарі для багатокласової класифікації?**
   *Відповідь:* `softmax`, оскільки вона нормує виходи у діапазон [0, 1] з сумою 1 (інтерпретуються як ймовірності класів).

3. **У чому відмінність між класифікацією та кластеризацією?**
   *Відповідь:* Класифікація — це Supervised Learning (є мітки класів $y$). Кластеризація — Unsupervised Learning (міток $y$ немає, групуємо об'єкти лише за ознаками $X$).